# Electric Motor Modelling
Aviary provides the capability to include electric motors as part of the propulsion system of the aircraft, including attaching one to a gearbox or propeller. 
In this section we will detail how the electric motor model works and point to a few examples of its implementation.

The Electric Motor model has a builder `MotorBuilder()` and two components, a `motor_premission()` componenet and a `motor_mission()` component. 
The `MotorBuilder()` contains the definitions of all of the inputs, outputs, and constraints required by the motor model. It also will need a motor map to be provided. 
Users can supply their own motor map or use the example map provided by `electric_motor_1800Nm_6000rpm.csv`. 

The example models all used a fixed RPM value, however, that is not a requirement.

## Motor Premission
Motor premission has the primary purpose of determining electric motor weight before the mission is flown.
This weight is a function of scale factor and motor torque. 
The scale factor scales the torque values on the motor map. 
This also allows the optimizer to always have throttle input values between 0-1.
The optimizer can be given control of the scale factor and it will scale the motor up and down, which will incur weight changes to the motor as well as scaling the motor maps.
During the calculations, the throttle value is set to 1 (max throttle) to determine the maximum torque. 
The maximum torque is then fed into the following equation:
$$\text{motor\_mass} = 0.3151 \times \text{max\_torque}^{0.748}$$
This equation is based on the work by [Duffy et. al.](https://www.researchgate.net/profile/Michael-Duffy-14/publication/326262943_Propulsion_Scaling_Methods_in_the_Era_of_Electric_Flight)
Motor mass relationship is based on continuous torque rating for aerospace motors shown in Figure 10 of the reference.

## Motor Mission
The motor mission takes as an input a scale factor, a motor map, a motor throttle (which can change during the mission), and an rpm.
This component outputs torque, maximum torque, and the electrical power demand.
All of these values are calculated for every point throughout the flight,.
The torque output can be sent to an attached propeller model.
The maximum torque is used to determine maximum excess power, enabling the users to place constraints on minimum motor torque. 
When combined with a propeller model, this will enable the user to place constraints such as 300ft/min climb capability during cruise, which is typical to ensure FAR requirementes for excess power in cruise are met.
The electrical power demanded by the motor is determined based on the calculated efficiency of the motor at a given torque / RPM combination.

## Motor Map
The MotorMap() component takes in 0-1 values for electric motor throttle and scales those values into 0-max_torque on the motor map. 
When paired with an RPM input value, this allows aviary to solve for the motor efficiency.
The map itself is constructed using a [structured meta model](https://openmdao.org/newdocs/versions/latest/features/building_blocks/components/metamodelstructured_comp.html). 
A structured meta model function is a smooth interpolation Component for data that exists on a regular, structured, grid.
The original maps were put together for a 746kw (1,000 hp) electric motor published by [Aretskin-Hariton et. al.](https://ntrs.nasa.gov/api/citations/20230016987/downloads/TTBW_SciTech_2024_Final_12_5_2023.pdf)
An electric motor map is required to be supplied in aviary options for the electric motor model to run successful. 
To do this, the user needs to set the aviary option: `Aircraft.Engine.Motor.DATA_FILE`. 
An example map is contained in `electric_motor_1800Nm_6000rpm.csv`.
This map is sent both to motor_premission and motor_mission via the motor builder.
![images/motor_efficiency_map_1800Nm_6000rpm.png](images/motor_efficiency_map_1800Nm_6000rpm.png)

## Electrical Power Consumption
An assessment of how much electric power the motor consumes during normal operations can be accessed by looking at `Dynamic.Vehicle.Propulsion.ELECTRIC_POWER_IN`. 
That will produce a vector of electric power usage during different parts of the mission.
This can serve as a starting point for how to size the electrical system.
However, the maximum possible electrical power required by the motor during exceptional conditions and off-nominal operations is not calculated by aviary automatically.

The maximum electric power and maximum torque the motor would consume while ensuring 300ft/min climb capability in cruise, is not necessarily achieved at maximum throttle. 
While this is usually the case in systems trying to shed mass, it is possible that it would be desirable to oversize and electric motor in order to run it at a more efficient parts of the motor map.
This in tern may support 300ft/min climb compliance without requiring the motor to be run at maximum torque. 
In those cases, the user would need to calculate for themselves what the maximum required torque was (not the maximum torque the motor was capable of producing which is already given by `Aircraft.Engine.Motor.TORQUE_MAX`). 
The maximum torque would need to be calculated at every point in the flight regime, and then, based on the RPM and resulting efficiencies, this would need to be translated into a maximum electric power.
This could then allow for electrical system sizing.
